### Inspect job postings

In [3]:
import os
import pandas as pd

job_postings = pd.read_csv('../data/job-postings/job_postings.csv')
job_postings["posting_id"] = [f"{i:05d}" for i in range(len(job_postings))]
print(job_postings.columns)

# Print the full text of the first 5 job postings for full inspection
for idx, jobpost in job_postings["jobpost"].head(3).items():
    print(f"\n\n--- Job Posting {idx} ---")
    print(jobpost)
    print("\n")


Index(['jobpost', 'date', 'Title', 'Company', 'AnnouncementCode', 'Term',
       'Eligibility', 'Audience', 'StartDate', 'Duration', 'Location',
       'JobDescription', 'JobRequirment', 'RequiredQual', 'Salary',
       'ApplicationP', 'OpeningDate', 'Deadline', 'Notes', 'AboutC', 'Attach',
       'Year', 'Month', 'IT', 'posting_id'],
      dtype='object')


--- Job Posting 0 ---
AMERIA Investment Consulting Company
JOB TITLE:  Chief Financial Officer
POSITION LOCATION: Yerevan, Armenia
JOB DESCRIPTION:   AMERIA Investment Consulting Company is seeking a
Chief Financial Officer. This position manages the company's fiscal and
administrative functions, provides highly responsible and technically
complex staff assistance to the Executive Director. The work performed
requires a high level of technical proficiency in financial management
and investment management, as well as management, supervisory, and
administrative skills.
JOB RESPONSIBILITIES:  
- Supervises financial management and adm

### Get all domains from the resume dataset


In [3]:
resumes_dir = "../data/cv-dataset/data"

# Get all domains from the resume dataset
domains = []
for dir in os.listdir(resumes_dir):
    if os.path.isdir(os.path.join(resumes_dir, dir)):
        domains.append(dir)

print(domains)

# Add aliases to the domains
domains_aliases = {
    "BUSINESS-DEVELOPMENT": "BUSINESS_DEVELOPMENT",
    "DIGITAL-MEDIA": "DIGITAL_MEDIA",
    "INFORMATION-TECHNOLOGY": "INFORMATION_TECHNOLOGY",
    "PUBLIC-RELATIONS": "PUBLIC_RELATIONS"
}


NameError: name 'os' is not defined

### Structurize 100 different postings

In [4]:
import sys
sys.path.insert(0, '..')
from baml_client.sync_client import b
from baml_client.types import JobPosting

def extract_job_posting(raw_job_posting: str) -> JobPosting: 
  # BAML's internal parser guarantees ExtractResume
  # to be always return a Resume type
  response = b.ExtractJobPostingLlamaCppPC(raw_job_posting)
  return response

In [5]:
import json
from pathlib import Path

output_base_dir = Path("../data/job-postings-processed")
output_base_dir.mkdir(parents=True, exist_ok=True)

allowed_domains = {
    "ACCOUNTANT", "ADVOCATE", "AGRICULTURE", "APPAREL", "ARTS", "AUTOMOBILE",
    "AVIATION", "BANKING", "BPO", "BUSINESS-DEVELOPMENT", "CHEF", "CONSTRUCTION",
    "CONSULTANT", "DESIGNER", "DIGITAL-MEDIA", "ENGINEERING", "FINANCE", "FITNESS",
    "HEALTHCARE", "HR", "INFORMATION-TECHNOLOGY", "PUBLIC-RELATIONS", "SALES", "TEACHER"
}

# Convert BAML enum-style or underscore names to the expected folder names.
domain_to_folder = {
    "BUSINESS_DEVELOPMENT": "BUSINESS-DEVELOPMENT",
    "DIGITAL_MEDIA": "DIGITAL-MEDIA",
    "INFORMATION_TECHNOLOGY": "INFORMATION-TECHNOLOGY",
    "PUBLIC_RELATIONS": "PUBLIC-RELATIONS",
}

posting_id_column = "posting_id" if "posting_id" in job_postings.columns else "position_id"

for idx, jobpost in job_postings["jobpost"].head(100).items():
    posting_id = str(job_postings.at[idx, posting_id_column]).zfill(5)

    # Skip if this posting was already processed in any domain folder.
    existing_matches = list(output_base_dir.glob(f"*/{posting_id}.json"))
    if existing_matches:
        continue

    try:
        extracted = extract_job_posting(jobpost)
    except Exception:
        continue

    # Support model objects (e.g. pydantic) and plain dict outputs.
    if hasattr(extracted, "model_dump"):
        payload = extracted.model_dump()
    elif hasattr(extracted, "dict"):
        payload = extracted.dict()
    else:
        payload = dict(extracted)

    raw_domain = str(payload.get("domain", "")).strip().upper()
    clean_domain = raw_domain.replace("JOBPOSTINGDOMAIN.", "")
    clean_domain = domains_aliases.get(clean_domain, clean_domain)
    clean_domain = clean_domain.replace("-", "_")

    folder_domain = domain_to_folder.get(clean_domain, clean_domain.replace("_", "-"))
    if folder_domain not in allowed_domains:
        continue

    payload["domain"] = folder_domain

    domain_dir = output_base_dir / folder_domain
    domain_dir.mkdir(parents=True, exist_ok=True)

    output_path = domain_dir / f"{posting_id}.json"

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"processed the posting with id: {posting_id}")



NameError: name 'job_postings' is not defined

### Structurize resumes (10 for each domain)

In [1]:
import sys
sys.path.insert(0, '..')
from baml_client.sync_client import b
from baml_client.types import Resume

def extract_resume(raw_resume: str) -> Resume: 
  # BAML's internal parser guarantees ExtractResume
  # to be always return a Resume type
  response = b.ExtractResumeLlamaCppPC(raw_resume)
  return response

In [2]:
import os

"""
Retrieve paths to up to 10 PDF files from each domain subfolder in data/cv-dataset/data.
"""

data_path = '../data/cv-dataset/data'
pdf_paths = []

# Loop over each domain subfolder and collect PDF paths
for domain_name in sorted(os.listdir(data_path)):
    domain_dir = os.path.join(data_path, domain_name)
    if not os.path.isdir(domain_dir):
        continue
    # Collect up to 10 pdf files in this domain folder
    domain_pdfs = [os.path.join(domain_dir, f) for f in os.listdir(domain_dir) if f.lower().endswith('.pdf')]
    domain_pdfs.sort()
    domain_pdfs = domain_pdfs[:10]
    pdf_paths.extend(domain_pdfs)

print("PDF paths retrieved")
print(pdf_paths[:5])

PDF paths retrieved
['../data/cv-dataset/data/ACCOUNTANT/10554236.pdf', '../data/cv-dataset/data/ACCOUNTANT/10674770.pdf', '../data/cv-dataset/data/ACCOUNTANT/11163645.pdf', '../data/cv-dataset/data/ACCOUNTANT/11759079.pdf', '../data/cv-dataset/data/ACCOUNTANT/12065211.pdf']


In [5]:

# Extract content from each file via fitz (PyMuPDF) and parse with extract_resume.
import io
import json
import signal
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path

import fitz  # PyMuPDF

source_data_root = Path("../data/cv-dataset/data").resolve()
resume_output_base_dir = Path("../data/cv-dataset-processed").resolve()
resume_output_base_dir.mkdir(parents=True, exist_ok=True)

if not pdf_paths:
    print("no pdf paths found")


class ExtractTimeout(Exception):
    pass


def _timeout_handler(signum, frame):
    raise ExtractTimeout("extract_resume timed out")


signal.signal(signal.SIGALRM, _timeout_handler)

processed = 0
skipped_existing = 0
failed = 0

for i, pdf_path in enumerate(pdf_paths, start=1):
    pdf_file = Path(pdf_path).resolve()

    try:
        relative_pdf_path = pdf_file.relative_to(source_data_root)
    except ValueError:
        # Fallback: keep at least domain/file structure if path roots differ.
        relative_pdf_path = Path(pdf_file.parent.name) / pdf_file.name

    output_path = (resume_output_base_dir / relative_pdf_path).with_suffix(".json")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        skipped_existing += 1
        print(f"[{i}/{len(pdf_paths)}] already exists, skipping: {output_path}")
        continue

    print(f"[{i}/{len(pdf_paths)}] reading: {relative_pdf_path}", flush=True)

    with fitz.open(pdf_file) as doc:
        raw_resume = "\n".join(page.get_text() for page in doc)

    preview_lines = [line.strip() for line in raw_resume.splitlines() if line.strip()][:5]
    if preview_lines:
        print("text preview:")
        for line in preview_lines:
            print(f"  {line[:180]}")
    else:
        print("text preview: <empty>")

    try:
        # Prevent one file from blocking the entire run.
        signal.alarm(60)
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            extracted = extract_resume(raw_resume)
    except ExtractTimeout:
        signal.alarm(0)
        failed += 1
        print(f"skipped due to timeout: {relative_pdf_path}")
        continue
    except Exception as e:
        signal.alarm(0)
        failed += 1
        print(f"skipped due to extraction error: {relative_pdf_path} ({e})")
        continue
    finally:
        signal.alarm(0)

    if hasattr(extracted, "model_dump"):
        payload = extracted.model_dump()
    elif hasattr(extracted, "dict"):
        payload = extracted.dict()
    else:
        payload = dict(extracted)

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    processed += 1
    parsed_name = str(payload.get("name", "")).strip() or pdf_file.stem
    parsed_domain = str(payload.get("domain", "")).strip() or relative_pdf_path.parent.name
    print(
        f"saved: {output_path} | parsed name={parsed_name} | parsed domain={parsed_domain}",
        flush=True,
    )

print(
    f"Done. processed={processed}, skipped_existing={skipped_existing}, failed={failed}, total={len(pdf_paths)}"
)


[1/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/10554236.json
[2/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/10674770.json
[3/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/11163645.json
[4/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/11759079.json
[5/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/12065211.json
[6/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/12202337.json
[7/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-processed/ACCOUNTANT/12338274.json
[8/240] already exists, skipping: /mnt/c/users/Tania/repos/CV-Job-Match-Maker/data/cv-dataset-pro